# U3-02 · Medidas y tipos de agrupamiento: mayoristas y clientes RFM

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 3, apartados 3.1 a 3.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar scikit-learn.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su preparación se explica en `LEEME.md`:
> Cardoso, M. (2013). *Wholesale customers*. https://doi.org/10.24432/C5030X ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Medidas de rendimiento con los mayoristas (3.2)

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score)

m = pd.read_csv("mayoristas.csv")
gasto = m.drop(columns=["canal", "region"])
X = StandardScaler().fit_transform(np.log1p(gasto))

def agrupar(X, k, semilla=0):
    return KMeans(k, n_init=10, random_state=semilla).fit_predict(X)

if __name__ == "__main__":
    print(gasto.describe().round(0).T[["mean", "50%", "max"]])
    for k in range(2, 7):
        e = agrupar(X, k)
        print(k, round(silhouette_score(X, e), 3),
              round(davies_bouldin_score(X, e), 2),
              round(calinski_harabasz_score(X, e), 1))
    e2 = agrupar(X, 2)
    print(pd.crosstab(e2, m.canal))
    print("ARI con el canal:",
          round(adjusted_rand_score(m.canal, e2), 3))
    # el mismo k-means sin logaritmo
    Xb = StandardScaler().fit_transform(gasto)
    eb = agrupar(Xb, 2)
    print("sin log:", np.bincount(eb),
          round(silhouette_score(Xb, eb), 3),
          round(adjusted_rand_score(m.canal, eb), 3))
    # estabilidad: reagrupar submuestras del 80 %
    rng = np.random.default_rng(1)
    for k in (2, 3, 4):
        ref, ari = agrupar(X, k), []
        for _ in range(20):
            i = rng.choice(len(X), int(.8 * len(X)), replace=False)
            ari.append(adjusted_rand_score(
                ref[i], agrupar(X[i], k, semilla=7)))
        print(k, "estabilidad", round(np.mean(ari), 3),
              round(np.min(ari), 3))

**Prueba.** Cambia `np.log1p(gasto)` por `gasto` en la línea que calcula `X` y vuelve a ejecutar la celda. ¿Qué número de grupos recomiendan ahora las medidas internas?

## 2. Segmentar a los clientes con RFM (3.3)

La última celda tarda unos segundos: ajusta siete mezclas gaussianas y dibuja el dendrograma.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

c = pd.read_csv("clientes_rfm.csv", index_col="cliente")
Z = StandardScaler().fit_transform(np.log1p(c))

def perfil(etiquetas):
    p = c.groupby(etiquetas).median()
    p["clientes"] = pd.Series(etiquetas).value_counts()
    return p.round(0)

if __name__ == "__main__":
    km = KMeans(4, n_init=10, random_state=0).fit_predict(Z)
    print(perfil(km))
    je = AgglomerativeClustering(4, linkage="ward").fit_predict(Z)
    print(perfil(je))
    print("ARI k-medias/jerárquico:",
          round(adjusted_rand_score(km, je), 3))
    for eps in (0.2, 0.3, 0.5):
        db = DBSCAN(eps=eps, min_samples=10).fit_predict(Z)
        print(eps, "grupos", db.max() + 1,
              "ruido", (db == -1).sum())
    print(perfil(db))                    # el último: eps = 0,5
    print("una sola compra:", (c.frecuencia == 1).sum())
    for k in range(2, 9):
        g = GaussianMixture(k, n_init=3, random_state=0).fit(Z)
        print(k, "BIC", round(g.bic(Z)))
    import matplotlib.pyplot as plt
    from scipy.cluster.hierarchy import linkage, dendrogram
    L = linkage(Z, method="ward")
    plt.figure(figsize=(4.4, 2.0))
    dendrogram(L, truncate_mode="lastp", p=30, no_labels=True,
               color_threshold=L[-3, 2],  # corte: 4 grupos
               above_threshold_color="black")
    plt.ylabel("distancia de Ward"); plt.show()  # plt.savefig("dendro.pdf")

**Prueba.** Repite DBSCAN solo con los clientes que tienen más de una compra (`Z[c.frecuencia.values > 1]`). ¿Cambian los grupos?

## 3. Mezclas gaussianas con los mayoristas (3.3)

In [ ]:
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture

bic = {k: GaussianMixture(k, n_init=3, random_state=0)
          .fit(X).bic(X) for k in range(1, 8)}
print({k: round(v) for k, v in bic.items()})
g = GaussianMixture(3, n_init=3, random_state=0).fit(X)
p = g.predict_proba(X)
e = p.argmax(axis=1)
print(pd.crosstab(e, m.canal))
print("pesos", g.weights_.round(2))
print("dudosos (máx. < 0,8):", (p.max(axis=1) < .8).sum())
gasto = m.drop(columns=["canal", "region"])
print(gasto.groupby(e).median().round(0).T)

---
## Listados de ampliación

Los listados siguientes completan los apartados del libro, en el mismo orden. Los que reutilizan un caso anterior lo importan de los ficheros `.py` que están junto a este cuaderno (por ejemplo, `from h_ventas import X, y`).

### Lo que la silueta no dice  ·  `l5_validez.py`

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

for dispersion in [0.5, 1.5, 3.0, 5.0]:
    X, verdad = make_blobs(600, centers=4, cluster_std=dispersion,
                           random_state=0)
    e = KMeans(4, n_init=10, random_state=0).fit_predict(X)
    print(f"dispersión {dispersion}: silueta "
          f"{silhouette_score(X, e):.3f}, ARI con la verdad "
          f"{adjusted_rand_score(verdad, e):.3f}")
# datos sin ningún grupo: uniformes en un cuadrado
U = np.random.default_rng(0).uniform(size=(600, 2))
for k in (2, 4, 8):
    e = KMeans(k, n_init=10, random_state=0).fit_predict(U)
    print(f"uniformes, k={k}: silueta {silhouette_score(U, e):.3f}")

### Varias medidas externas  ·  `l7_externas.py`

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score,
                             adjusted_mutual_info_score,
                             homogeneity_score, completeness_score)
from l_mayoristas import X, m

for k in [2, 3, 4, 6]:
    e = KMeans(k, n_init=10, random_state=0).fit_predict(X)
    print(f"k={k}: ARI {adjusted_rand_score(m.canal, e):.3f}  "
          f"AMI {adjusted_mutual_info_score(m.canal, e):.3f}  "
          f"homog. {homogeneity_score(m.canal, e):.3f}  "
          f"compl. {completeness_score(m.canal, e):.3f}")

### Elegir el número de grupos  ·  `l2_codo.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from l_mayoristas import X                     # 440 x 6, log y escala

ks = range(1, 11)
inercia, silueta = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=0).fit(X)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X, km.labels_)
                   if k > 1 else None)
for k, i, s in zip(ks, inercia, silueta):
    caida = "" if k == 1 else f"  baja {1 - i / inercia[k - 2]:.0%}"
    print(f"k={k:<2} inercia {i:7.1f}{caida}"
          + ("" if s is None else f"  silueta {s:.3f}"))
fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.8))
a.plot(ks, inercia, "o-", ms=3); a.set_xlabel("k")
a.set_title("Inercia (codo)")
b.plot(ks[1:], silueta[1:], "o-", ms=3, c="C1"); b.set_xlabel("k")
b.set_title("Silueta media")
plt.show()  # fig.savefig("codo.pdf")

### La silueta grupo a grupo  ·  `l3_silueta.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples
from l_mayoristas import X, m

e = KMeans(3, n_init=10, random_state=0).fit_predict(X)
s = silhouette_samples(X, e)
fig, eje = plt.subplots(figsize=(4.4, 2.2))
y0 = 0
for g in range(3):
    sg = np.sort(s[e == g])
    eje.fill_betweenx(np.arange(y0, y0 + len(sg)), 0, sg,
                      color=f"C{g}", lw=0)
    eje.text(0.6, y0 + len(sg) / 2, f"grupo {g}", va="center",
             ha="left", fontsize=6)
    print(f"grupo {g}: {len(sg)} clientes, silueta media "
          f"{sg.mean():.3f}, negativos {(sg < 0).sum()}")
    y0 += len(sg) + 10
eje.axvline(s.mean(), ls="--", c="0.5", lw=0.8)
eje.set_yticks([]); eje.set_xlabel("silueta de cada cliente")
eje.set_xlim(-0.3, 0.75); plt.show()  # fig.savefig("silueta.pdf")
print(f"silueta media global: {s.mean():.3f}")

### Describir los grupos  ·  `l4_perfiles.py`

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from l_mayoristas import X, gasto, m

e = KMeans(3, n_init=10, random_state=0).fit_predict(X)
perfil = gasto.groupby(e).median().round(-1).astype(int)
perfil["clientes"] = pd.Series(e).value_counts().sort_index()
print(perfil.T.to_string())
total = gasto.sum(axis=1)
print("gasto anual total, mediana por grupo:",
      total.groupby(e).median().round(-2).astype(int).tolist())
print(pd.crosstab(e, m.canal.map({1: "Horeca", 2: "minorista"}),
                  rownames=["grupo"], colnames=[""]))

### ¿Qué grupos se repiten?  ·  `m8_estabilidad.py`

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from m_rfm import Z, perfil                  # 5878 clientes RFM

ref = KMeans(4, n_init=10, random_state=0).fit_predict(Z)
rng = np.random.default_rng(0)
jac = {g: [] for g in range(4)}
for _ in range(20):
    i = np.unique(rng.integers(0, len(Z), len(Z)))  # bootstrap
    e = KMeans(4, n_init=10, random_state=1).fit_predict(Z[i])
    for g in range(4):
        A = set(i[ref[i] == g])
        # el grupo nuevo que más se parece al de referencia
        jac[g].append(max(len(A & set(i[e == h]))
                          / len(A | set(i[e == h]))
                          for h in range(4)))
p = perfil(ref)
for g in range(4):
    print(f"grupo {g} (recencia {p.recencia[g]:.0f}, frecuencia "
          f"{p.frecuencia[g]:.0f}): Jaccard medio "
          f"{np.mean(jac[g]):.2f}, mínimo {np.min(jac[g]):.2f}")

### La escala decide los grupos  ·  `l6_escalado.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                   RobustScaler)
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from m_rfm import c                       # 5878 clientes RFM

variantes = {
    "sin escalar": c.to_numpy(float),
    "estándar": StandardScaler().fit_transform(c),
    "mín-máx": MinMaxScaler().fit_transform(c),
    "robusto": RobustScaler().fit_transform(c),
    "log + estándar": StandardScaler().fit_transform(np.log1p(c))}
e = {n: KMeans(4, n_init=10, random_state=0).fit_predict(X)
     for n, X in variantes.items()}
for n in variantes:
    tam = sorted(np.bincount(e[n]).tolist(), reverse=True)
    ari = adjusted_rand_score(e[n], e["log + estándar"])
    print(f"{n:<15} tamaños {str(tam):<24} ARI con log {ari:.2f}")

### k-medias desde cero  ·  `m5_desde_cero.py`

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from l_mayoristas import X                     # 440 x 6

def kmedias(X, k, semilla=0, iter_max=100):
    rng = np.random.default_rng(semilla)
    centros = X[rng.choice(len(X), k, replace=False)]
    for i in range(iter_max):
        # 1) asignar: distancia de cada punto a cada centro
        d = ((X[:, None, :] - centros[None]) ** 2).sum(axis=2)
        grupo = d.argmin(axis=1)
        # 2) mover cada centro a la media de sus puntos
        nuevos = np.array([X[grupo == j].mean(axis=0)
                           for j in range(k)])
        if np.allclose(nuevos, centros):
            break
        centros = nuevos
    inercia = d[np.arange(len(X)), grupo].sum()
    return grupo, inercia, i + 1

mejor = min((kmedias(X, 3, s) for s in range(10)),
            key=lambda r: r[1])
print(f"propio: inercia {mejor[1]:.1f} en {mejor[2]} iteraciones")
km = KMeans(3, n_init=10, random_state=0).fit(X)
print(f"scikit-learn: inercia {km.inertia_:.1f} "
      f"en {km.n_iter_} iteraciones")

### RFM clásico por quintiles  ·  `m7_quintiles.py`

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from m_rfm import c, Z                       # 5878 clientes RFM

# puntuación clásica: 1 a 5 por quintiles (5 = mejor)
r = pd.qcut(c.recencia, 5, labels=[5, 4, 3, 2, 1]).astype(int)
f = pd.qcut(c.frecuencia.rank(method="first"), 5,
            labels=[1, 2, 3, 4, 5]).astype(int)
m = pd.qcut(c.importe, 5, labels=[1, 2, 3, 4, 5]).astype(int)
total = r + f + m                            # de 3 a 15
print("celdas RFM distintas usadas:",
      (r.astype(str) + f.astype(str) + m.astype(str)).nunique(),
      "de 125")
print("clientes 555:", ((r == 5) & (f == 5) & (m == 5)).sum(),
      "| clientes 111:", ((r == 1) & (f == 1) & (m == 1)).sum())
km = KMeans(4, n_init=10, random_state=0).fit_predict(Z)
tabla = pd.crosstab(km, pd.cut(total, [2, 6, 9, 12, 15],
                               labels=["3-6", "7-9", "10-12",
                                       "13-15"]))
tabla.index.name, tabla.columns.name = "k-medias", "puntos"
print(tabla.to_string())

### El enlace del agrupamiento jerárquico  ·  `m2_enlaces.py`

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from m_rfm import Z                          # 5878 clientes RFM

e = {}
for enlace in ["single", "average", "complete", "ward"]:
    e[enlace] = AgglomerativeClustering(
        4, linkage=enlace).fit_predict(Z)
    tam = sorted(np.bincount(e[enlace]).tolist(), reverse=True)
    print(f"{enlace:<9} tamaños {tam}")
for enlace in ["single", "average", "complete"]:
    ari = adjusted_rand_score(e[enlace], e["ward"])
    print(f"ARI {enlace} frente a ward: {ari:.3f}")

### Elegir eps en DBSCAN  ·  `m9_eps.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from m_rfm import Z                          # 5878 clientes RFM

k = 10                                       # = min_samples
dist, _ = NearestNeighbors(n_neighbors=k).fit(Z).kneighbors(Z)
d = np.sort(dist[:, -1])                     # distancia al 10.º
for q in [50, 90, 95, 99]:
    print(f"percentil {q}: {np.percentile(d, q):.2f}")
for eps in [0.2, 0.35, 0.6]:
    e = DBSCAN(eps=eps, min_samples=k).fit_predict(Z)
    print(f"eps {eps}: {e.max() + 1} grupos, ruido {(e == -1).sum()}")
fig, eje = plt.subplots(figsize=(4.4, 1.8))
eje.plot(d); eje.set_ylabel(f"distancia al vecino {k}")
eje.set_xlabel("clientes ordenados"); eje.set_ylim(0, 1.5)
plt.show()  # fig.savefig("eps.pdf")

### Densidad jerárquica: HDBSCAN  ·  `m3_hdbscan.py`

In [ ]:
from sklearn.cluster import HDBSCAN
from m_rfm import Z, perfil                  # 5878 clientes RFM

for minimo in [25, 100, 300]:
    e = HDBSCAN(min_cluster_size=minimo, copy=True).fit_predict(Z)
    print(f"grupo mínimo {minimo:>3}: {e.max() + 1} grupos,"
          f" ruido {(e == -1).sum()}")
    if minimo == 100:
        print(perfil(e).to_string())

### Grupos con forma: el agrupamiento espectral  ·  `m4_circulos.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.cluster import KMeans, SpectralClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score

X, y = make_circles(600, factor=0.45, noise=0.06, random_state=0)
modelos = {
    "k-medias": KMeans(2, n_init=10, random_state=0),
    "espectral": SpectralClustering(2, affinity="nearest_neighbors",
                                    n_neighbors=15, random_state=0),
    "DBSCAN": DBSCAN(eps=0.15, min_samples=5)}
fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.6))
for eje, (nombre, m) in zip(ejes, modelos.items()):
    e = m.fit_predict(X)
    print(f"{nombre:<10} ARI {adjusted_rand_score(y, e):.3f}  "
          f"grupos {len(set(e) - {-1})}")
    eje.scatter(X[:, 0], X[:, 1], c=e, s=2, cmap="tab10",
                vmin=-1, vmax=9)
    eje.set_title(nombre); eje.set_axis_off()
    eje.set_aspect("equal")
plt.show()  # fig.savefig("circulos.pdf")

### Clientes atípicos  ·  `n2_anomalias.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from l_mayoristas import X, gasto, m          # log y escala

bosque = IsolationForest(n_estimators=300, contamination=0.02,
                         random_state=0).fit(X)
p_bosque = -bosque.score_samples(X)          # más alto, más raro
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof.fit(X)
p_lof = -lof.negative_outlier_factor_
raros_b = set(pd.Series(p_bosque).nlargest(9).index)
raros_l = set(pd.Series(p_lof).nlargest(9).index)
print("señalados por los dos:", sorted(raros_b & raros_l))
print("correlación de las puntuaciones:",
      round(pd.Series(p_bosque).corr(pd.Series(p_lof),
                                     method="spearman"), 2))
r = (gasto / gasto.median()).round(2)        # veces la mediana
r.columns = [c[:5] for c in r.columns]      # nombres cortos
print(r.loc[sorted(raros_b & raros_l)].assign(
    canal=m.canal).to_string())

### La forma de las componentes  ·  `n3_covarianzas.py`

In [ ]:
from sklearn.mixture import GaussianMixture
from l_mayoristas import X                    # 440 x 6

for tipo in ["spherical", "diag", "tied", "full"]:
    bics = {k: GaussianMixture(k, covariance_type=tipo, n_init=5,
                               random_state=0).fit(X).bic(X)
            for k in range(1, 7)}
    mejor = min(bics, key=bics.get)
    print(f"{tipo:<9} mejor k={mejor}, BIC {bics[mejor]:.0f}")

**Ejercicio.** Prueba `GaussianMixture(3, covariance_type="diag")`. ¿Qué número de componentes elige ahora el BIC?

> Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7